# Training Notebook

**Goal**
Train a chunked diffusion trajectory model from cached Waymo shards and produce reproducible checkpoint artifacts.

**What this notebook does**
Walks through configuration, cache validation, data loading, model/training utilities, optimization loop, diagnostics, and export.

**Inputs**
- Cached shards under `./waymo_cache_v2/{train,val}`
- PyTorch runtime (GPU preferred)

**Output / interpretation**
- Resumable training checkpoint
- Best EMA checkpoint
- Exported inference checkpoint with normalization/token tables


## 0) Prerequisites

- Run `python download.py` in the terminal to obtain the cached dataset before training

If any issues occured, read `download_readme.md`


## 1) Configuration and Global Constants

**Goal**
Declare a single source of truth for training hyperparameters (This can be tuned using grid search or other hyperparameter tuning methods).

**What this section does**
Defines batch/training settings, feature dimensions, diffusion parameters, and filesystem paths.


In [ ]:
import logging
import math
import random
from collections import defaultdict
from contextlib import nullcontext
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, IterableDataset
from tqdm.auto import tqdm

SEED = 24
BATCH_SIZE = 192
GRAD_ACCUM_STEPS = 3
NUM_WORKERS = 4
SMOKE_MAX_BATCHES = 60
FULL_EPOCHS = 10
RUN_FULL_TRAINING = True
USE_TORCH_COMPILE = False
RESUME_FROM_LATEST = True

PAST_STEPS = 10
CURRENT_TIME_INDEX = 10
HISTORY_STEPS = CURRENT_TIME_INDEX + 1
FUTURE_STEPS = 80
TOTAL_SCENE_STEPS = HISTORY_STEPS + FUTURE_STEPS
NEIGHBORS_K = 6

assert HISTORY_STEPS == 11
assert TOTAL_SCENE_STEPS == 91

HIST_DIM = 13
NBR_DIM = 10
MAP_DIM = 5
STATIC_DIM = 7
TARGET_DIM = FUTURE_STEPS * 4
COND_DIM = 256

DIFFUSION_T = 200
DIFFUSION_SAMPLE_STEPS = 50
GUIDANCE_SCALE = 1.2
EMA_DECAY = 0.999
CFG_DROPOUT = 0.1

POS_TOKEN_COUNT = 512
TRAJ_TOKEN_COUNT = 1024
TRAJ_TOKEN_STEPS = 5
TOKEN_BUILD_MAX_SHARDS = 32
TOKEN_BUILD_MAX_SAMPLES = 200_000
TOKEN_KMEANS_ITERS = 18

BASE_LR = 1e-4
WEIGHT_DECAY = 1e-4
WARMUP_RATIO = 0.05
GRAD_CLIP_NORM = 1.0
CLOSED_LOOP_AUG_PROB = 0.35
HIST_POS_NOISE_STD = 0.15
HIST_VEL_NOISE_STD = 0.10
NBR_POS_NOISE_STD = 0.25
NBR_VEL_NOISE_STD = 0.15
REWARD_BASELINE_MOMENTUM = 0.99
HIGH_SPEED_STEP_THRESHOLD = 2.6
HIGH_SPEED_REG_MULT = 1.35

LOSS_WEIGHTS = {
    "w_diff": 1.0,
    "w_smooth": 0.02,
    "w_yaw": 0.01,
    "w_offroad": 0.02,
    "w_collision": 0.02,
    "w_momentum_long": 0.02,
    "w_momentum_lat": 0.03,
    "w_lane_align": 0.01,
    "w_reward": 0.01,
    "w_pos_ce": 0.02,
    "w_traj_ce": 0.02,
}

CACHE_ROOT = Path("./waymo_cache_v2")
TRAIN_CACHE_DIR = CACHE_ROOT / "train"
VAL_CACHE_DIR = CACHE_ROOT / "val"
CHECKPOINT_DIR = Path("./checkpoints")
TRAINING_CHECKPOINT_PATH = CHECKPOINT_DIR / "latest_training.pt"
BEST_EMA_CHECKPOINT_PATH = CHECKPOINT_DIR / "best_ema.pt"
LOG_DIR = Path("./logs")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_ENABLED = DEVICE.type == "cuda"

def seed_everything(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


## 2) Cache Validation and Data Loading

**What this section does**
Checks schema/shape assumptions, builds iterable datasets/loaders, and estimates target normalization stats.

**Inputs**
- `samples_*.pt` shards and optional split manifests.

**Output / interpretation**
- Confirmed cache contract and ready-to-iterate dataloaders.


In [ ]:
TRAIN_SHARDS = sorted(str(p) for p in TRAIN_CACHE_DIR.glob("samples_*.pt"))
VAL_SHARDS = sorted(str(p) for p in VAL_CACHE_DIR.glob("samples_*.pt"))

assert TRAIN_SHARDS, f"No training shards found in {TRAIN_CACHE_DIR.resolve()}"
assert VAL_SHARDS, f"No validation shards found in {VAL_CACHE_DIR.resolve()}"

print("Train shards:", len(TRAIN_SHARDS))
print("Val shards:", len(VAL_SHARDS))
print("First train shard:", TRAIN_SHARDS[0])
print("First val shard:", VAL_SHARDS[0])

In [ ]:
def load_shard(shard_path: str) -> dict:
    return torch.load(shard_path, map_location="cpu")


def maybe_load_manifest(cache_dir: Path) -> dict | None:
    manifest_path = cache_dir / "manifest.pt"
    if not manifest_path.exists():
        return None
    return torch.load(manifest_path, map_location="cpu")


sample_shard = load_shard(TRAIN_SHARDS[0])
required_shard_keys = {"hist", "nbr", "map", "static", "target", "masks", "meta"}
required_mask_keys = {"hist_valid", "target_valid", "nbr_valid", "map_valid"}

assert required_shard_keys.issubset(set(sample_shard.keys()))
assert required_mask_keys.issubset(set(sample_shard["masks"].keys()))

for key in ["hist", "nbr", "map", "static", "target", "meta"]:
    assert sample_shard[key].dtype == torch.float32, f"Unexpected dtype for {key}: {sample_shard[key].dtype}"

loaded_history_steps = int(sample_shard["hist"].shape[1])
loaded_future_steps = int(sample_shard["target"].shape[1])
assert loaded_history_steps == HISTORY_STEPS, (
    f"Cached hist shape is {loaded_history_steps}, expected {HISTORY_STEPS}. "
    "Rebuild waymo_cache_v2 with download.py before retraining."
)
assert loaded_future_steps == FUTURE_STEPS, (
    f"Cached target shape is {loaded_future_steps}, expected {FUTURE_STEPS}. "
    "Rebuild waymo_cache_v2 with download.py before retraining."
)

hist_valid_shape = tuple(sample_shard["masks"]["hist_valid"].shape)
target_valid_shape = tuple(sample_shard["masks"]["target_valid"].shape)
assert hist_valid_shape[1] == HISTORY_STEPS
assert target_valid_shape[1] == FUTURE_STEPS

anchor_indices = sample_shard["meta"][:, 2].to(torch.int64)
assert int(anchor_indices.min()) >= CURRENT_TIME_INDEX
assert int(anchor_indices.max()) <= 70
print("Anchor times in first train shard:", sorted(torch.unique(anchor_indices).tolist())[:20])

train_manifest = maybe_load_manifest(TRAIN_CACHE_DIR)
val_manifest = maybe_load_manifest(VAL_CACHE_DIR)
for split_name, manifest in [("train", train_manifest), ("val", val_manifest)]:
    if manifest is None:
        print(f"{split_name} manifest: not found")
        continue

    print(
        f"{split_name} manifest:",
        {k: manifest[k] for k in ["H", "F", "K", "total_samples", "multi_anchor", "anchor_stride", "min_future_valid"] if k in manifest},
    )
    assert int(manifest["H"]) == HISTORY_STEPS
    assert int(manifest["F"]) == FUTURE_STEPS
    assert int(manifest["K"]) == NEIGHBORS_K

    if bool(manifest.get("multi_anchor", False)):
        anchor_counts = manifest.get("anchor_counts", {})
        expected_anchors = list(range(CURRENT_TIME_INDEX, 71, int(manifest.get("anchor_stride", 5))))
        missing = [a for a in expected_anchors if int(anchor_counts.get(a, 0)) <= 0]
        assert not missing, f"Missing anchor_t values in {split_name} manifest: {missing}"

print("hist shape:", tuple(sample_shard["hist"].shape))
print("nbr shape:", tuple(sample_shard["nbr"].shape))
print("map shape:", tuple(sample_shard["map"].shape))
print("static shape:", tuple(sample_shard["static"].shape))
print("target shape:", tuple(sample_shard["target"].shape))
print("meta shape:", tuple(sample_shard["meta"].shape))


In [ ]:
class CachedShardIterableDataset(IterableDataset):
    def __init__(self, shard_paths: list[str], shuffle: bool = True, seed: int = SEED):
        super().__init__()
        self.shard_paths = list(shard_paths)
        self.shuffle = shuffle
        self.seed = seed

    def __iter__(self):
        worker_info = torch.utils.data.get_worker_info()
        worker_id = worker_info.id if worker_info is not None else 0

        rng = random.Random(self.seed + worker_id)
        shard_paths = list(self.shard_paths)
        if self.shuffle:
            rng.shuffle(shard_paths)

        for shard_path in shard_paths:
            shard = load_shard(shard_path)
            n = int(shard["hist"].shape[0])
            indices = list(range(n))
            if self.shuffle:
                rng.shuffle(indices)

            for idx in indices:
                yield {
                    "hist": shard["hist"][idx],
                    "nbr": shard["nbr"][idx],
                    "map": shard["map"][idx],
                    "static": shard["static"][idx],
                    "target": shard["target"][idx],
                    "masks": {
                        "hist_valid": shard["masks"]["hist_valid"][idx],
                        "target_valid": shard["masks"]["target_valid"][idx],
                        "nbr_valid": shard["masks"]["nbr_valid"][idx],
                        "map_valid": shard["masks"]["map_valid"][idx],
                    },
                    "meta": shard["meta"][idx],
                }

def load_cached_dataset(
    shard_paths: list[str],
    batch_size: int,
    shuffle: bool = True,
    num_workers: int = NUM_WORKERS,
) -> DataLoader:
    dataset = CachedShardIterableDataset(shard_paths, shuffle=shuffle, seed=SEED)
    pin_memory = DEVICE.type == "cuda"
    worker_count = max(0, int(num_workers))
    return DataLoader(
        dataset,
        batch_size=batch_size,
        num_workers=worker_count,
        pin_memory=pin_memory,
        persistent_workers=(worker_count > 0),
    )

train_loader = load_cached_dataset(TRAIN_SHARDS, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = load_cached_dataset(VAL_SHARDS, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
print({k: tuple(v.shape) if hasattr(v, "shape") else type(v) for k, v in batch.items() if k != "masks"})
print({k: tuple(v.shape) for k, v in batch["masks"].items()})


In [ ]:
def move_batch_to_device(batch: dict, device: torch.device) -> dict:
    return {
        "hist": batch["hist"].to(device, non_blocking=True),
        "nbr": batch["nbr"].to(device, non_blocking=True),
        "map": batch["map"].to(device, non_blocking=True),
        "static": batch["static"].to(device, non_blocking=True),
        "target": batch["target"].to(device, non_blocking=True),
        "masks": {
            "hist_valid": batch["masks"]["hist_valid"].to(device, non_blocking=True),
            "target_valid": batch["masks"]["target_valid"].to(device, non_blocking=True),
            "nbr_valid": batch["masks"]["nbr_valid"].to(device, non_blocking=True),
            "map_valid": batch["masks"]["map_valid"].to(device, non_blocking=True),
        },
        "meta": batch["meta"].to(device, non_blocking=True),
    }

def estimate_target_stats(shard_paths: list[str], max_shards: int = 32) -> tuple[torch.Tensor, torch.Tensor]:
    sum_x = torch.zeros(4, dtype=torch.float64)
    sum_x2 = torch.zeros(4, dtype=torch.float64)
    count = torch.zeros(4, dtype=torch.float64)

    for shard_path in shard_paths[:max_shards]:
        shard = load_shard(shard_path)
        target = shard["target"].double()
        mask = shard["masks"]["target_valid"].double().unsqueeze(-1)

        sum_x += (target * mask).sum(dim=(0, 1))
        sum_x2 += ((target ** 2) * mask).sum(dim=(0, 1))
        count += mask.sum(dim=(0, 1)).squeeze(-1)

    mean = (sum_x / count.clamp_min(1.0)).float()
    var = (sum_x2 / count.clamp_min(1.0)).float() - mean ** 2
    std = torch.sqrt(torch.clamp(var, min=1e-6))
    return mean, std

target_mean, target_std = estimate_target_stats(TRAIN_SHARDS, max_shards=min(32, len(TRAIN_SHARDS)))
print("Target mean:", target_mean)
print("Target std:", target_std)

## 3) Diffusion Core Imports

**What this section does**
Imports model/token/sampling helpers from `notebooks_lib.diffusion_core`.

**Inputs**
- `notebooks_lib/diffusion_core.py`

In [ ]:
from notebooks_lib.diffusion_core import (
    make_cosine_schedule,
    build_token_tables_from_shards,
    nearest_token_indices,
    ConditionEncoder,
    ChunkDiffusionModel,
    update_ema,
    copy_model_params,
    zero_condition_like,
    apply_cfg_dropout,
    q_sample,
    sample_future_chunk,
)


In [ ]:
def unwrap_model(model: nn.Module) -> nn.Module:
    return getattr(model, "_orig_mod", model)


def setup_logger(name: str = "train", log_dir: Path = LOG_DIR) -> logging.Logger:
    log_dir.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger(name)
    logger.setLevel(logging.INFO)
    logger.propagate = False

    for handler in list(logger.handlers):
        logger.removeHandler(handler)
        handler.close()

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")

    stream_handler = logging.StreamHandler()
    stream_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_dir / f"train_{timestamp}.log")
    file_handler.setFormatter(formatter)

    logger.addHandler(stream_handler)
    logger.addHandler(file_handler)
    return logger


def save_training_checkpoint(
    checkpoint_path: Path,
    epoch: int,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    history: list[dict],
    token_tables: dict | None = None,
) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "epoch": epoch,
        "model_state_dict": unwrap_model(model).state_dict(),
        "ema_state_dict": unwrap_model(ema_model).state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "history": history,
        "target_mean": target_mean.detach().cpu(),
        "target_std": target_std.detach().cpu(),
        "current_time_index": CURRENT_TIME_INDEX,
        "history_steps": HISTORY_STEPS,
        "future_steps": FUTURE_STEPS,
        "total_scene_steps": TOTAL_SCENE_STEPS,
        "neighbors_k": NEIGHBORS_K,
        "diffusion_t": DIFFUSION_T,
        "sample_steps": DIFFUSION_SAMPLE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "ema_decay": EMA_DECAY,
    }
    if token_tables is not None:
        payload["position_tokens"] = token_tables.get("position_tokens", None)
        payload["trajectory_tokens"] = token_tables.get("trajectory_tokens", None)

    torch.save(payload, checkpoint_path)


def save_best_ema_checkpoint(
    checkpoint_path: Path,
    epoch: int,
    ema_model: nn.Module,
    val_loss: float,
    token_tables: dict | None = None,
) -> None:
    checkpoint_path.parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "epoch": epoch,
        "val_loss": float(val_loss),
        "ema_state_dict": unwrap_model(ema_model).state_dict(),
        "target_mean": target_mean.detach().cpu(),
        "target_std": target_std.detach().cpu(),
        "current_time_index": CURRENT_TIME_INDEX,
        "history_steps": HISTORY_STEPS,
        "future_steps": FUTURE_STEPS,
        "neighbors_k": NEIGHBORS_K,
        "diffusion_t": DIFFUSION_T,
        "sample_steps": DIFFUSION_SAMPLE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
    }
    if token_tables is not None:
        payload["position_tokens"] = token_tables.get("position_tokens", None)
        payload["trajectory_tokens"] = token_tables.get("trajectory_tokens", None)
    torch.save(payload, checkpoint_path)


def load_training_checkpoint(
    checkpoint_path: Path,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    device: torch.device,
    enabled: bool = True,
    logger: logging.Logger | None = None,
    expected_full_epochs: int | None = None,
    reset_if_finished: bool = True,
    load_optimizer: bool = True,
    load_scaler: bool = True,
) -> tuple[int, list[dict], dict | None]:
    if (not enabled) or (not checkpoint_path.exists()):
        return 1, [], None

    try:
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model_msg = unwrap_model(model).load_state_dict(checkpoint["model_state_dict"], strict=False)
        ema_msg = unwrap_model(ema_model).load_state_dict(checkpoint["ema_state_dict"], strict=False)
        if logger is not None:
            if model_msg.missing_keys or model_msg.unexpected_keys:
                logger.info("Model checkpoint mismatch: missing=%s unexpected=%s", model_msg.missing_keys, model_msg.unexpected_keys)
            if ema_msg.missing_keys or ema_msg.unexpected_keys:
                logger.info("EMA checkpoint mismatch: missing=%s unexpected=%s", ema_msg.missing_keys, ema_msg.unexpected_keys)
        if load_optimizer and "optimizer_state_dict" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
        if load_scaler and "scaler_state_dict" in checkpoint:
            scaler.load_state_dict(checkpoint["scaler_state_dict"])
        history = checkpoint.get("history", [])
        last_epoch = int(checkpoint.get("epoch", 0))
    except Exception:
        if logger is not None:
            logger.exception("Failed to load checkpoint %s", checkpoint_path)
        return 1, [], None

    if expected_full_epochs is not None and reset_if_finished and last_epoch >= expected_full_epochs:
        if logger is not None:
            logger.info("Checkpoint already finished at epoch %s. Starting fresh.", last_epoch)
        return 1, [], checkpoint

    if logger is not None:
        logger.info("Resumed %s checkpoint from epoch %s", checkpoint_path, last_epoch)
    return last_epoch + 1, history, checkpoint


## 4) Training and Evaluation Functions

**Goal**
Define optimization-time losses, regularizers, and epoch loops.

**What this section does**
Provides finite-check guards, realism regularizers, train/eval loops, and scheduling helpers.

**Inputs**
- Model, schedule, token tables, and dataloaders.

**Output / interpretation**
- Reusable training primitives called by the execution block.


In [ ]:
def _is_finite_tensor(t: torch.Tensor) -> bool:
    return bool(torch.isfinite(t).all())


def _is_finite_batch(batch: dict) -> bool:
    tensors = [batch["hist"], batch["nbr"], batch["map"], batch["static"], batch["target"], batch["meta"]]
    tensors += [
        batch["masks"]["hist_valid"],
        batch["masks"]["target_valid"],
        batch["masks"]["nbr_valid"],
        batch["masks"]["map_valid"],
    ]
    return all(_is_finite_tensor(t) for t in tensors)


def _compute_aux_token_losses(
    pos_logits: torch.Tensor | None,
    traj_logits: torch.Tensor | None,
    target: torch.Tensor,
    target_valid: torch.Tensor,
    token_tables: dict | None,
) -> tuple[torch.Tensor, torch.Tensor]:
    if token_tables is None or pos_logits is None or traj_logits is None:
        zero = target.new_tensor(0.0)
        return zero, zero

    pos_tokens = token_tables["position_tokens"].to(target.device)
    traj_tokens = token_tables["trajectory_tokens"].to(target.device)

    b = target.shape[0]
    pos_target = target[:, 0, :2]
    pos_idx = nearest_token_indices(pos_target, pos_tokens)
    pos_loss = F.cross_entropy(pos_logits, pos_idx)

    valid_traj = target_valid[:, :TRAJ_TOKEN_STEPS].sum(dim=1) >= TRAJ_TOKEN_STEPS
    if bool(valid_traj.any()):
        traj_target = target[valid_traj, :TRAJ_TOKEN_STEPS, :2].reshape(-1, TRAJ_TOKEN_STEPS * 2)
        traj_idx = nearest_token_indices(traj_target, traj_tokens.reshape(traj_tokens.shape[0], -1))
        traj_loss = F.cross_entropy(traj_logits[valid_traj], traj_idx)
    else:
        traj_loss = target.new_tensor(0.0)

    return pos_loss, traj_loss


def _masked_mean(values: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    mask = mask.to(values.dtype)
    return (values * mask).sum() / mask.sum().clamp_min(1.0)


In [ ]:
def _compute_realism_regularizers(x0_pred: torch.Tensor, cond: dict) -> dict[str, torch.Tensor]:
    # x0_pred is in local ego frame with channels [x, y, sin(yaw), cos(yaw)]
    xy = x0_pred[..., :2]
    sin_h = x0_pred[..., 2]
    cos_h = x0_pred[..., 3]

    yaw = torch.atan2(sin_h, cos_h + 1e-6)
    dxy = xy[:, 1:, :] - xy[:, :-1, :]

    # Momentum decomposition: longitudinal (forward/back) and lateral (left/right)
    heading = torch.stack([torch.cos(yaw[:, :-1]), torch.sin(yaw[:, :-1])], dim=-1)
    lateral_axis = torch.stack([-heading[..., 1], heading[..., 0]], dim=-1)

    long_vel = (dxy * heading).sum(dim=-1)
    lat_vel = (dxy * lateral_axis).sum(dim=-1)

    zero_scalar = x0_pred.new_tensor(0.0)

    if long_vel.shape[1] > 1:
        long_acc = long_vel[:, 1:] - long_vel[:, :-1]
        lat_acc = lat_vel[:, 1:] - lat_vel[:, :-1]
    else:
        long_acc = long_vel.new_zeros((long_vel.shape[0], 0))
        lat_acc = lat_vel.new_zeros((lat_vel.shape[0], 0))

    smooth_loss = (
        F.smooth_l1_loss(long_acc, torch.zeros_like(long_acc), reduction="mean")
        + F.smooth_l1_loss(lat_acc, torch.zeros_like(lat_acc), reduction="mean")
    ) if long_acc.numel() > 0 else zero_scalar

    yaw_delta = torch.atan2(torch.sin(yaw[:, 1:] - yaw[:, :-1]), torch.cos(yaw[:, 1:] - yaw[:, :-1]))
    yaw_loss = F.smooth_l1_loss(yaw_delta, torch.zeros_like(yaw_delta), reduction="mean") if yaw_delta.numel() > 0 else zero_scalar

    momentum_long_loss = (
        F.smooth_l1_loss(long_acc[:, 1:] - long_acc[:, :-1], torch.zeros_like(long_acc[:, 1:] - long_acc[:, :-1]), reduction="mean")
        if long_acc.shape[1] > 1
        else zero_scalar
    )
    momentum_lat_loss = (
        0.5 * F.smooth_l1_loss(lat_vel, torch.zeros_like(lat_vel), reduction="mean")
        + F.smooth_l1_loss(lat_acc, torch.zeros_like(lat_acc), reduction="mean")
    ) if lat_vel.numel() > 0 else zero_scalar

    # RL-style road-boundary penalty (normalized), using map edge distance as local boundary proxy.
    map_valid = cond["masks"]["map_valid"].squeeze(-1)
    edge_dist = cond["map"][:, 3].clamp(min=0.0, max=120.0).unsqueeze(1)
    road_radius = (edge_dist + 5.0).clamp_min(8.0)
    radial = torch.linalg.norm(xy, dim=-1)
    boundary_violation = F.relu(radial - road_radius)
    offroad_loss = _masked_mean(boundary_violation / road_radius.clamp_min(1.0), map_valid.unsqueeze(1).expand_as(boundary_violation))

    # Neighbor collision proxy on first predicted step.
    nbr_xy = cond["nbr"][..., :2]
    nbr_valid = cond["masks"]["nbr_valid"]
    pred0 = xy[:, 0, :].unsqueeze(1)
    nbr_dist = torch.linalg.norm(pred0 - nbr_xy, dim=-1)
    collision_raw = F.relu(2.5 - nbr_dist) / 2.5
    collision_loss = _masked_mean(collision_raw, nbr_valid)

    # Lane-alignment penalty using map lane direction in local frame.
    lane_sin = cond["map"][:, 1]
    lane_cos = cond["map"][:, 2]
    lane_dir = torch.stack([lane_cos, lane_sin], dim=-1)
    lane_dir = F.normalize(lane_dir, dim=-1)

    if dxy.shape[1] > 0:
        vel0 = dxy[:, 0, :]
        speed0 = torch.linalg.norm(vel0, dim=-1)
        vel0_dir = F.normalize(vel0 + 1e-6, dim=-1)
        align = (vel0_dir * lane_dir).sum(dim=-1)
        lane_align_pen = F.relu(0.2 - align) * (speed0 > 0.1).to(align.dtype)
        lane_align_loss = _masked_mean(lane_align_pen, map_valid)
    else:
        lane_align_loss = zero_scalar

    # Safe reward (RLFTSim-style dense reward shaping inspiration, without policy gradient instability).
    safe_reward = 1.0 - (
        2.0 * offroad_loss
        + 1.5 * collision_loss
        + 0.5 * momentum_lat_loss
        + 0.5 * momentum_long_loss
        + 0.25 * yaw_loss
        + 0.25 * lane_align_loss
    )
    safe_reward = torch.clamp(safe_reward, min=-2.0, max=1.0)

    # Long-tail high-speed indicator (comBOT insight): regularize safety terms more when speed is high.
    high_speed_frac = (cond["static"][:, -1] >= HIGH_SPEED_STEP_THRESHOLD).float().mean()

    return {
        "smooth": smooth_loss,
        "yaw": yaw_loss,
        "offroad": offroad_loss,
        "collision": collision_loss,
        "momentum_long": momentum_long_loss,
        "momentum_lat": momentum_lat_loss,
        "lane_align": lane_align_loss,
        "safe_reward": safe_reward,
        "high_speed_frac": high_speed_frac,
    }


def apply_closed_loop_augmentation(cond: dict, p: float = CLOSED_LOOP_AUG_PROB) -> dict:
    # UniMM-inspired distribution-shift mitigation: perturb conditioning during training.
    if p <= 0.0:
        return cond
    if float(torch.rand(1, device=cond["hist"].device).item()) >= p:
        return cond

    hist_valid = cond["masks"]["hist_valid"].unsqueeze(-1)
    nbr_valid = cond["masks"]["nbr_valid"].unsqueeze(-1)

    hist = cond["hist"].clone()
    nbr = cond["nbr"].clone()

    hist[:, :, :2] += torch.randn_like(hist[:, :, :2]) * HIST_POS_NOISE_STD * hist_valid
    hist[:, :, 2:4] += torch.randn_like(hist[:, :, 2:4]) * HIST_VEL_NOISE_STD * hist_valid

    nbr[:, :, :2] += torch.randn_like(nbr[:, :, :2]) * NBR_POS_NOISE_STD * nbr_valid
    nbr[:, :, 2:4] += torch.randn_like(nbr[:, :, 2:4]) * NBR_VEL_NOISE_STD * nbr_valid

    cond_aug = {
        "hist": hist,
        "nbr": nbr,
        "map": cond["map"],
        "static": cond["static"],
        "masks": cond["masks"],
    }
    return cond_aug


In [ ]:
def train_one_epoch(
    loader: DataLoader,
    model: nn.Module,
    ema_model: nn.Module,
    optimizer: torch.optim.Optimizer,
    scaler: torch.amp.GradScaler,
    lr_scheduler: torch.optim.lr_scheduler._LRScheduler,
    schedule: dict,
    target_mean: torch.Tensor,
    target_std: torch.Tensor,
    epoch: int,
    max_batches: int | None = None,
    loss_weights: dict | None = None,
    finiteness_guard: bool = True,
    token_tables: dict | None = None,
    grad_accum_steps: int = GRAD_ACCUM_STEPS,
) -> dict:
    model.train()
    losses = []
    T = int(schedule["T"].item())
    progress = tqdm(loader, total=max_batches, desc=f"train epoch {epoch}", leave=False)
    stats = defaultdict(int)

    if loss_weights is None:
        loss_weights = LOSS_WEIGHTS

    component_sums = defaultdict(float)
    reward_baseline = None

    optimizer.zero_grad(set_to_none=True)
    pending_backward_steps = 0

    for batch_idx, batch in enumerate(progress):
        if max_batches is not None and batch_idx >= max_batches:
            break
        stats["seen_batches"] += 1

        batch = move_batch_to_device(batch, DEVICE)
        if finiteness_guard and (not _is_finite_batch(batch)):
            stats["skipped_nonfinite_batch"] += 1
            continue

        cond = {
            "hist": batch["hist"],
            "nbr": batch["nbr"],
            "map": batch["map"],
            "static": batch["static"],
            "masks": batch["masks"],
        }
        cond_shift = apply_closed_loop_augmentation(cond, p=CLOSED_LOOP_AUG_PROB)
        cond_drop = apply_cfg_dropout(cond_shift, p_drop=CFG_DROPOUT)

        target = batch["target"]
        target_valid = batch["masks"]["target_valid"]
        target_mask = target_valid.unsqueeze(-1).expand(-1, -1, 4)
        x0 = (target - target_mean.view(1, 1, 4)) / target_std.view(1, 1, 4)

        if finiteness_guard and (not _is_finite_tensor(x0)):
            stats["skipped_nonfinite_x0"] += 1
            continue

        x0_flat = x0.reshape(x0.shape[0], -1)
        t = torch.randint(0, T, (x0_flat.shape[0],), device=DEVICE)
        x_t, noise = q_sample(x0_flat, t, schedule)
        t_norm = (t.float() / float(T)).unsqueeze(-1)

        ctx = torch.autocast(device_type="cuda", enabled=AMP_ENABLED) if AMP_ENABLED else nullcontext()
        with ctx:
            pred_noise, pos_logits, traj_logits, _ = model.forward_with_aux(x_t, t_norm, cond_drop)
            if finiteness_guard and (not _is_finite_tensor(pred_noise)):
                stats["skipped_nonfinite_pred"] += 1
                continue

            flat_mask = target_mask.reshape(target_mask.shape[0], -1)
            diff_loss = (((pred_noise - noise) ** 2) * flat_mask).sum() / flat_mask.sum().clamp_min(1.0)

            sqrt_ab = schedule["sqrt_alphas_cumprod"][t].unsqueeze(-1)
            sqrt_omb = schedule["sqrt_one_minus_alphas_cumprod"][t].unsqueeze(-1)
            x0_pred_flat = (x_t - sqrt_omb * pred_noise) / sqrt_ab.clamp_min(1e-6)
            x0_pred = x0_pred_flat.view(target.shape[0], FUTURE_STEPS, 4)
            x0_pred = x0_pred * target_std.view(1, 1, 4) + target_mean.view(1, 1, 4)

            regs = _compute_realism_regularizers(x0_pred, cond)
            pos_ce, traj_ce = _compute_aux_token_losses(pos_logits, traj_logits, target, target_valid, token_tables)

            high_speed_mult = 1.0 + (HIGH_SPEED_REG_MULT - 1.0) * regs["high_speed_frac"]

            batch_reward = float(regs["safe_reward"].detach().item())
            if reward_baseline is None:
                reward_baseline = batch_reward
            reward_baseline = REWARD_BASELINE_MOMENTUM * reward_baseline + (1.0 - REWARD_BASELINE_MOMENTUM) * batch_reward
            reward_adv = regs["safe_reward"] - x0_pred.new_tensor(reward_baseline)
            reward_loss = -reward_adv

            total_loss = (
                loss_weights["w_diff"] * diff_loss
                + loss_weights["w_smooth"] * regs["smooth"]
                + loss_weights["w_yaw"] * regs["yaw"]
                + high_speed_mult * loss_weights["w_offroad"] * regs["offroad"]
                + high_speed_mult * loss_weights["w_collision"] * regs["collision"]
                + loss_weights["w_momentum_long"] * regs["momentum_long"]
                + high_speed_mult * loss_weights["w_momentum_lat"] * regs["momentum_lat"]
                + high_speed_mult * loss_weights["w_lane_align"] * regs["lane_align"]
                + loss_weights["w_reward"] * reward_loss
                + loss_weights["w_pos_ce"] * pos_ce
                + loss_weights["w_traj_ce"] * traj_ce
            )

        if finiteness_guard and (not torch.isfinite(total_loss)):
            stats["skipped_nonfinite_loss"] += 1
            continue

        scaler.scale(total_loss / float(grad_accum_steps)).backward()
        pending_backward_steps += 1
        stats["finite_batches"] += 1

        if pending_backward_steps >= grad_accum_steps:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            pending_backward_steps = 0
            stats["optimizer_steps"] += 1
            update_ema(ema_model, unwrap_model(model), decay=EMA_DECAY)
            lr_scheduler.step()

        loss_value = float(total_loss.item())
        losses.append(loss_value)

        component_sums["diff"] += float(diff_loss.item())
        component_sums["smooth"] += float(regs["smooth"].item())
        component_sums["yaw"] += float(regs["yaw"].item())
        component_sums["offroad"] += float(regs["offroad"].item())
        component_sums["collision"] += float(regs["collision"].item())
        component_sums["momentum_long"] += float(regs["momentum_long"].item())
        component_sums["momentum_lat"] += float(regs["momentum_lat"].item())
        component_sums["lane_align"] += float(regs["lane_align"].item())
        component_sums["reward_loss"] += float(reward_loss.item())
        component_sums["safe_reward"] += float(regs["safe_reward"].item())
        component_sums["high_speed_frac"] += float(regs["high_speed_frac"].item())
        component_sums["pos_ce"] += float(pos_ce.item())
        component_sums["traj_ce"] += float(traj_ce.item())

        progress.set_postfix(loss=f"{loss_value:.4f}", avg=f"{float(np.mean(losses)):.4f}", hs=f"{float(regs['high_speed_frac'].item()):.2f}", finite=f"{stats['finite_batches']}/{stats['seen_batches']}")

    if pending_backward_steps > 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        stats["optimizer_steps"] += 1
        update_ema(ema_model, unwrap_model(model), decay=EMA_DECAY)
        lr_scheduler.step()

    progress.close()
    finite_loss_ratio = float(stats["finite_batches"]) / float(max(1, stats["seen_batches"]))
    denom = float(max(1, stats["finite_batches"]))
    components_avg = {k: (v / denom) for k, v in component_sums.items()}
    return {
        "losses": losses,
        "avg_loss": float(np.mean(losses)) if losses else float("nan"),
        "finite_loss_ratio": finite_loss_ratio,
        "stats": dict(stats),
        "components": components_avg,
    }


In [ ]:
def evaluate_one_epoch(
    loader: DataLoader,
    model: nn.Module,
    schedule: dict,
    target_mean: torch.Tensor,
    target_std: torch.Tensor,
    epoch: int,
    max_batches: int | None = None,
    loss_weights: dict | None = None,
    finiteness_guard: bool = True,
    token_tables: dict | None = None,
) -> dict:
    model.eval()
    losses = []
    T = int(schedule["T"].item())
    progress = tqdm(loader, total=max_batches, desc=f"eval epoch {epoch}", leave=False)
    stats = defaultdict(int)

    if loss_weights is None:
        loss_weights = LOSS_WEIGHTS

    component_sums = defaultdict(float)

    for batch_idx, batch in enumerate(progress):
        if max_batches is not None and batch_idx >= max_batches:
            break
        stats["seen_batches"] += 1

        batch = move_batch_to_device(batch, DEVICE)
        if finiteness_guard and (not _is_finite_batch(batch)):
            stats["skipped_nonfinite_batch"] += 1
            continue

        cond = {
            "hist": batch["hist"],
            "nbr": batch["nbr"],
            "map": batch["map"],
            "static": batch["static"],
            "masks": batch["masks"],
        }

        target = batch["target"]
        target_valid = batch["masks"]["target_valid"]
        target_mask = target_valid.unsqueeze(-1).expand(-1, -1, 4)
        x0 = (target - target_mean.view(1, 1, 4)) / target_std.view(1, 1, 4)
        x0_flat = x0.reshape(x0.shape[0], -1)

        t = torch.randint(0, T, (x0_flat.shape[0],), device=DEVICE)
        x_t, noise = q_sample(x0_flat, t, schedule)
        t_norm = (t.float() / float(T)).unsqueeze(-1)

        pred_noise, pos_logits, traj_logits, _ = model.forward_with_aux(x_t, t_norm, cond)
        flat_mask = target_mask.reshape(target_mask.shape[0], -1)
        diff_loss = (((pred_noise - noise) ** 2) * flat_mask).sum() / flat_mask.sum().clamp_min(1.0)

        sqrt_ab = schedule["sqrt_alphas_cumprod"][t].unsqueeze(-1)
        sqrt_omb = schedule["sqrt_one_minus_alphas_cumprod"][t].unsqueeze(-1)
        x0_pred_flat = (x_t - sqrt_omb * pred_noise) / sqrt_ab.clamp_min(1e-6)
        x0_pred = x0_pred_flat.view(target.shape[0], FUTURE_STEPS, 4)
        x0_pred = x0_pred * target_std.view(1, 1, 4) + target_mean.view(1, 1, 4)

        regs = _compute_realism_regularizers(x0_pred, cond)
        pos_ce, traj_ce = _compute_aux_token_losses(pos_logits, traj_logits, target, target_valid, token_tables)
        high_speed_mult = 1.0 + (HIGH_SPEED_REG_MULT - 1.0) * regs["high_speed_frac"]
        reward_loss = -regs["safe_reward"]
        total_loss = (
            loss_weights["w_diff"] * diff_loss
            + loss_weights["w_smooth"] * regs["smooth"]
            + loss_weights["w_yaw"] * regs["yaw"]
            + high_speed_mult * loss_weights["w_offroad"] * regs["offroad"]
            + high_speed_mult * loss_weights["w_collision"] * regs["collision"]
            + loss_weights["w_momentum_long"] * regs["momentum_long"]
            + high_speed_mult * loss_weights["w_momentum_lat"] * regs["momentum_lat"]
            + high_speed_mult * loss_weights["w_lane_align"] * regs["lane_align"]
            + loss_weights["w_reward"] * reward_loss
            + loss_weights["w_pos_ce"] * pos_ce
            + loss_weights["w_traj_ce"] * traj_ce
        )

        if finiteness_guard and (not torch.isfinite(total_loss)):
            stats["skipped_nonfinite_loss"] += 1
            continue

        loss_value = float(total_loss.item())
        losses.append(loss_value)
        stats["finite_batches"] += 1

        component_sums["diff"] += float(diff_loss.item())
        component_sums["smooth"] += float(regs["smooth"].item())
        component_sums["yaw"] += float(regs["yaw"].item())
        component_sums["offroad"] += float(regs["offroad"].item())
        component_sums["collision"] += float(regs["collision"].item())
        component_sums["momentum_long"] += float(regs["momentum_long"].item())
        component_sums["momentum_lat"] += float(regs["momentum_lat"].item())
        component_sums["lane_align"] += float(regs["lane_align"].item())
        component_sums["reward_loss"] += float(reward_loss.item())
        component_sums["safe_reward"] += float(regs["safe_reward"].item())
        component_sums["high_speed_frac"] += float(regs["high_speed_frac"].item())
        component_sums["pos_ce"] += float(pos_ce.item())
        component_sums["traj_ce"] += float(traj_ce.item())

        progress.set_postfix(loss=f"{loss_value:.4f}", avg=f"{float(np.mean(losses)):.4f}")

    progress.close()
    finite_loss_ratio = float(stats["finite_batches"]) / float(max(1, stats["seen_batches"]))
    denom = float(max(1, stats["finite_batches"]))
    components_avg = {k: (v / denom) for k, v in component_sums.items()}
    return {
        "losses": losses,
        "avg_loss": float(np.mean(losses)) if losses else float("nan"),
        "finite_loss_ratio": finite_loss_ratio,
        "stats": dict(stats),
        "components": components_avg,
    }


## 5) Execute Training Pipeline

**Goal**
Run smoke checks, full-epoch optimization, and checkpointing.

**What this section does**
Initializes logger/model/optimizer, optionally resumes, runs training epochs, and tracks best EMA.

**Inputs**
- Prior sections plus cache-derived token tables.

**Output / interpretation**
- Logged history, latest checkpoint, and best-validation EMA checkpoint.


In [ ]:
TRAIN_LOGGER = setup_logger()
TRAIN_LOGGER.info("Initializing training pipeline")
TRAIN_LOGGER.info(
    "Config: batch_size=%s grad_accum=%s num_workers=%s full_epochs=%s resume=%s compile=%s device=%s",
    BATCH_SIZE,
    GRAD_ACCUM_STEPS,
    NUM_WORKERS,
    FULL_EPOCHS,
    RESUME_FROM_LATEST,
    USE_TORCH_COMPILE,
    DEVICE,
)

TRAIN_LOGGER.info("Building token tables from cache")
TOKEN_TABLES = build_token_tables_from_shards(
    TRAIN_SHARDS,
    pos_k=POS_TOKEN_COUNT,
    traj_k=TRAJ_TOKEN_COUNT,
    traj_steps=TRAJ_TOKEN_STEPS,
    max_shards=min(TOKEN_BUILD_MAX_SHARDS, len(TRAIN_SHARDS)),
    max_samples=TOKEN_BUILD_MAX_SAMPLES,
)
TOKEN_TABLES = {k: v.float() for k, v in TOKEN_TABLES.items()}

pos_vocab_size = int(TOKEN_TABLES["position_tokens"].shape[0])
traj_vocab_size = int(TOKEN_TABLES["trajectory_tokens"].shape[0])

base_model = ChunkDiffusionModel(pos_vocab_size=pos_vocab_size, traj_vocab_size=traj_vocab_size).to(DEVICE)
model = torch.compile(base_model) if (USE_TORCH_COMPILE and hasattr(torch, "compile")) else base_model
ema_model = ChunkDiffusionModel(pos_vocab_size=pos_vocab_size, traj_vocab_size=traj_vocab_size).to(DEVICE)
copy_model_params(base_model, ema_model)
ema_model.eval()

schedule = make_cosine_schedule(DIFFUSION_T, DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)

train_total_samples = int(train_manifest.get("total_samples", 0)) if train_manifest is not None else 0
if train_total_samples <= 0:
    train_total_samples = BATCH_SIZE * 1000
steps_per_epoch = max(1, math.ceil(train_total_samples / BATCH_SIZE / GRAD_ACCUM_STEPS))
total_optimizer_steps = max(1, steps_per_epoch * FULL_EPOCHS)
warmup_steps = max(10, int(WARMUP_RATIO * total_optimizer_steps))

def _lr_lambda(step: int) -> float:
    if step < warmup_steps:
        return float(step + 1) / float(max(1, warmup_steps))
    progress = float(step - warmup_steps) / float(max(1, total_optimizer_steps - warmup_steps))
    return 0.5 * (1.0 + math.cos(math.pi * progress))

lr_scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=_lr_lambda)

start_epoch, history, resumed_checkpoint = load_training_checkpoint(
    TRAINING_CHECKPOINT_PATH,
    model=model,
    ema_model=ema_model,
    optimizer=optimizer,
    scaler=scaler,
    device=DEVICE,
    enabled=RUN_FULL_TRAINING and RESUME_FROM_LATEST,
    logger=TRAIN_LOGGER,
    expected_full_epochs=FULL_EPOCHS,
)

if resumed_checkpoint is not None:
    if resumed_checkpoint.get("position_tokens") is not None and resumed_checkpoint.get("trajectory_tokens") is not None:
        TOKEN_TABLES["position_tokens"] = resumed_checkpoint["position_tokens"].float()
        TOKEN_TABLES["trajectory_tokens"] = resumed_checkpoint["trajectory_tokens"].float()

run_smoke = (not RUN_FULL_TRAINING) or (resumed_checkpoint is None and start_epoch == 1)
if run_smoke:
    TRAIN_LOGGER.info("Running smoke training for %s batches", SMOKE_MAX_BATCHES)
    smoke_stats = train_one_epoch(
        train_loader,
        model,
        ema_model,
        optimizer,
        scaler,
        lr_scheduler,
        schedule,
        target_mean.to(DEVICE),
        target_std.to(DEVICE),
        epoch=0,
        max_batches=SMOKE_MAX_BATCHES,
        loss_weights=LOSS_WEIGHTS,
        finiteness_guard=True,
        token_tables=TOKEN_TABLES,
        grad_accum_steps=GRAD_ACCUM_STEPS,
    )
    smoke_losses = smoke_stats["losses"]
    if len(smoke_losses) >= 10:
        head = float(np.mean(smoke_losses[:5]))
        tail = float(np.mean(smoke_losses[-5:]))
        TRAIN_LOGGER.info("Smoke loss head=%.4f tail=%.4f finite_ratio=%.4f", head, tail, smoke_stats["finite_loss_ratio"])

    val_stats = evaluate_one_epoch(
        val_loader,
        ema_model,
        schedule,
        target_mean.to(DEVICE),
        target_std.to(DEVICE),
        epoch=0,
        max_batches=20,
        loss_weights=LOSS_WEIGHTS,
        finiteness_guard=True,
        token_tables=TOKEN_TABLES,
    )
    TRAIN_LOGGER.info("Validation loss (20 batches): %.4f finite_ratio=%.4f", val_stats["avg_loss"], val_stats["finite_loss_ratio"])
else:
    TRAIN_LOGGER.info("Skipping smoke training because a resumable checkpoint was loaded")

history = list(history)
best_val = min([h.get("val_loss", float("inf")) for h in history], default=float("inf"))

if RUN_FULL_TRAINING:
    if start_epoch > FULL_EPOCHS:
        TRAIN_LOGGER.info("Training already complete at epoch %s", start_epoch - 1)
    for epoch in range(start_epoch, FULL_EPOCHS + 1):
        TRAIN_LOGGER.info("Starting epoch %s/%s", epoch, FULL_EPOCHS)

        train_stats = train_one_epoch(
            train_loader,
            model,
            ema_model,
            optimizer,
            scaler,
            lr_scheduler,
            schedule,
            target_mean.to(DEVICE),
            target_std.to(DEVICE),
            epoch=epoch,
            max_batches=None,
            loss_weights=LOSS_WEIGHTS,
            finiteness_guard=True,
            token_tables=TOKEN_TABLES,
            grad_accum_steps=GRAD_ACCUM_STEPS,
        )

        val_stats = evaluate_one_epoch(
            val_loader,
            ema_model,
            schedule,
            target_mean.to(DEVICE),
            target_std.to(DEVICE),
                        epoch=epoch,
            max_batches=40,
            loss_weights=LOSS_WEIGHTS,
            finiteness_guard=True,
            token_tables=TOKEN_TABLES,
        )

        summary = {
            "epoch": epoch,
            "train_loss": float(train_stats["avg_loss"]),
            "train_finite_loss_ratio": float(train_stats["finite_loss_ratio"]),
            "train_diff": float(train_stats.get("components", {}).get("diff", float("nan"))),
            "train_offroad": float(train_stats.get("components", {}).get("offroad", float("nan"))),
            "train_collision": float(train_stats.get("components", {}).get("collision", float("nan"))),
            "train_momentum_long": float(train_stats.get("components", {}).get("momentum_long", float("nan"))),
            "train_momentum_lat": float(train_stats.get("components", {}).get("momentum_lat", float("nan"))),
            "train_safe_reward": float(train_stats.get("components", {}).get("safe_reward", float("nan"))),
            "train_high_speed_frac": float(train_stats.get("components", {}).get("high_speed_frac", float("nan"))),
            "val_loss": float(val_stats["avg_loss"]),
            "val_finite_loss_ratio": float(val_stats["finite_loss_ratio"]),
            "val_diff": float(val_stats.get("components", {}).get("diff", float("nan"))),
            "val_offroad": float(val_stats.get("components", {}).get("offroad", float("nan"))),
            "val_collision": float(val_stats.get("components", {}).get("collision", float("nan"))),
            "val_momentum_long": float(val_stats.get("components", {}).get("momentum_long", float("nan"))),
            "val_momentum_lat": float(val_stats.get("components", {}).get("momentum_lat", float("nan"))),
            "val_safe_reward": float(val_stats.get("components", {}).get("safe_reward", float("nan"))),
            "val_high_speed_frac": float(val_stats.get("components", {}).get("high_speed_frac", float("nan"))),
            "lr": float(optimizer.param_groups[0]["lr"]),
            "optimizer_steps": int(train_stats["stats"].get("optimizer_steps", 0)),
        }
        history.append(summary)
        TRAIN_LOGGER.info("Epoch summary: %s", summary)

        save_training_checkpoint(
            TRAINING_CHECKPOINT_PATH,
            epoch=epoch,
            model=model,
            ema_model=ema_model,
            optimizer=optimizer,
            scaler=scaler,
            history=history,
            token_tables=TOKEN_TABLES,
        )
        TRAIN_LOGGER.info("Saved training checkpoint to %s", TRAINING_CHECKPOINT_PATH)

        if np.isfinite(summary["val_loss"]) and summary["val_loss"] < best_val:
            best_val = summary["val_loss"]
            save_best_ema_checkpoint(
                BEST_EMA_CHECKPOINT_PATH,
                epoch=epoch,
                ema_model=ema_model,
                val_loss=best_val,
                token_tables=TOKEN_TABLES,
            )
            TRAIN_LOGGER.info("Saved best EMA checkpoint to %s (val_loss=%.6f)", BEST_EMA_CHECKPOINT_PATH, best_val)

DIFFUSION_CFG = {
    "schedule": schedule,
    "sample_steps": DIFFUSION_SAMPLE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "target_mean": target_mean.to(DEVICE),
    "target_std": target_std.to(DEVICE),
    "ema_model": ema_model,
    "position_tokens": TOKEN_TABLES["position_tokens"].to(DEVICE),
    "trajectory_tokens": TOKEN_TABLES["trajectory_tokens"].to(DEVICE),
}

TRAIN_LOGGER.info("Model parameter device: %s", next(unwrap_model(model).parameters()).device)
print("Model parameter device:", next(unwrap_model(model).parameters()).device)


## 6) Diagnostics and Export

**Goal**
Visualize prediction behavior and export a portable checkpoint for inference.

**What this section does**
Computes masked ADE/FDE diagnostics and saves packed model assets.

**Inputs**
- Trained model + EMA + normalization/token artifacts.

**Output / interpretation**
- Publish-ready checkpoint (`chunk_diffusion_pytorch_only.pt`).


In [ ]:
@torch.no_grad()
def build_cond_from_batch(batch: dict) -> dict:
    return {
        "hist": batch["hist"],
        "nbr": batch["nbr"],
        "map": batch["map"],
        "static": batch["static"],
        "masks": batch["masks"],
    }

sample_batch = next(iter(val_loader))
sample_batch = move_batch_to_device(sample_batch, DEVICE)
sample_cond = build_cond_from_batch(sample_batch)
pred_future = sample_future_chunk(model, sample_cond, DIFFUSION_CFG, use_ema=True).detach().cpu()
target_future = sample_batch["target"].detach().cpu()
target_valid = sample_batch["masks"]["target_valid"].detach().cpu()
hist = sample_batch["hist"].detach().cpu()
hist_valid = sample_batch["masks"]["hist_valid"].detach().cpu()

idx = 0
hist_xy = hist[idx, hist_valid[idx] > 0, :2].numpy()
pred_xy = pred_future[idx, :, :2].numpy()
target_xy = target_future[idx, target_valid[idx] > 0, :2].numpy()

plt.figure(figsize=(6, 6))
if len(hist_xy) > 0:
    plt.plot(hist_xy[:, 0], hist_xy[:, 1], marker="o", label="history")
if len(target_xy) > 0:
    plt.plot(target_xy[:, 0], target_xy[:, 1], marker="o", label="target future")
plt.plot(pred_xy[:, 0], pred_xy[:, 1], marker="x", label="pred future")
plt.axhline(0.0, color="gray", linewidth=0.5)
plt.axvline(0.0, color="gray", linewidth=0.5)
plt.gca().set_aspect("equal", adjustable="box")
plt.legend()
plt.title("Local-frame trajectory sample")
plt.show()

## Full-Horizon Diagnostics

`newWaymo.ipynb` is useful as a reminder that the model should be checked on the full 91-step scene layout, not only on loss. The diagnostics below keep your structured model but reconstruct a full local 91-step trajectory, compute masked ADE/FDE on a validation batch, and visualize one sample over the complete horizon.

In [ ]:
@torch.no_grad()
def masked_ade_fde(pred_future: torch.Tensor, target_future: torch.Tensor, target_valid: torch.Tensor) -> tuple[float, float]:
    pred_xy = pred_future[..., :2]
    target_xy = target_future[..., :2]
    valid_mask = target_valid > 0
    dist = torch.linalg.norm(pred_xy - target_xy, dim=-1)

    ade = float(dist[valid_mask].mean().item()) if bool(valid_mask.any()) else float("nan")

    row_ids = torch.arange(valid_mask.shape[0], device=valid_mask.device)
    step_ids = torch.arange(valid_mask.shape[1], device=valid_mask.device).unsqueeze(0).expand_as(valid_mask)
    last_valid_idx = torch.where(valid_mask, step_ids, torch.full_like(step_ids, -1)).max(dim=1).values
    has_valid = last_valid_idx >= 0
    if bool(has_valid.any()):
        fde = float(dist[row_ids[has_valid], last_valid_idx[has_valid]].mean().item())
    else:
        fde = float("nan")
    return ade, fde


def reconstruct_full_local_xy(
    hist_xy: torch.Tensor,
    hist_valid: torch.Tensor,
    future_xy: torch.Tensor,
    future_valid: torch.Tensor,
) -> tuple[np.ndarray, np.ndarray]:
    full_xy = np.full((TOTAL_SCENE_STEPS, 2), np.nan, dtype=np.float32)
    full_valid = np.zeros((TOTAL_SCENE_STEPS,), dtype=bool)

    hist_xy_np = hist_xy.detach().cpu().numpy()
    hist_valid_np = (hist_valid.detach().cpu().numpy() > 0)
    future_xy_np = future_xy.detach().cpu().numpy()
    future_valid_np = (future_valid.detach().cpu().numpy() > 0)

    full_xy[:HISTORY_STEPS] = hist_xy_np
    full_valid[:HISTORY_STEPS] = hist_valid_np
    full_xy[HISTORY_STEPS:] = future_xy_np
    full_valid[HISTORY_STEPS:] = future_valid_np
    return full_xy, full_valid


sample_batch = next(iter(val_loader))
sample_batch = move_batch_to_device(sample_batch, DEVICE)
sample_cond = build_cond_from_batch(sample_batch)
pred_future = sample_future_chunk(model, sample_cond, DIFFUSION_CFG, use_ema=True).detach().cpu()
target_future = sample_batch["target"].detach().cpu()
target_valid = sample_batch["masks"]["target_valid"].detach().cpu()
hist = sample_batch["hist"].detach().cpu()
hist_valid = sample_batch["masks"]["hist_valid"].detach().cpu()

ade, fde = masked_ade_fde(pred_future, target_future, target_valid)
print(f"Masked ADE: {ade:.4f}")
print(f"Masked FDE: {fde:.4f}")

idx = 0
full_gt_xy, full_gt_valid = reconstruct_full_local_xy(
    hist[idx, :, :2],
    hist_valid[idx],
    target_future[idx, :, :2],
    target_valid[idx],
)
full_pred_xy, full_pred_valid = reconstruct_full_local_xy(
    hist[idx, :, :2],
    hist_valid[idx],
    pred_future[idx, :, :2],
    target_valid[idx],
)

time_axis = np.arange(-PAST_STEPS, FUTURE_STEPS + 1)

plt.figure(figsize=(7, 7))
if np.any(full_gt_valid[:HISTORY_STEPS]):
    plt.plot(
        full_gt_xy[:HISTORY_STEPS, 0],
        full_gt_xy[:HISTORY_STEPS, 1],
        marker="o",
        linewidth=2,
        label="history / current",
    )
if np.any(full_gt_valid[HISTORY_STEPS:]):
    plt.plot(
        full_gt_xy[HISTORY_STEPS:, 0],
        full_gt_xy[HISTORY_STEPS:, 1],
        marker="o",
        linewidth=2,
        label="ground-truth future",
    )
if np.any(full_pred_valid[HISTORY_STEPS:]):
    plt.plot(
        full_pred_xy[HISTORY_STEPS:, 0],
        full_pred_xy[HISTORY_STEPS:, 1],
        marker="x",
        linewidth=2,
        label="predicted future",
    )

plt.scatter(
    [full_gt_xy[CURRENT_TIME_INDEX, 0]],
    [full_gt_xy[CURRENT_TIME_INDEX, 1]],
    s=80,
    c="black",
    label="current timestep",
)
plt.axhline(0.0, color="gray", linewidth=0.5)
plt.axvline(0.0, color="gray", linewidth=0.5)
plt.gca().set_aspect("equal", adjustable="box")
plt.title(f"Full 91-step local trajectory sample (t in [{time_axis[0]}, {time_axis[-1]}])")
plt.legend()
plt.show()

In [ ]:
SAVE_PATH = "./chunk_diffusion_pytorch_only.pt"

torch.save(
    {
        "model_state_dict": unwrap_model(model).state_dict(),
        "ema_state_dict": unwrap_model(ema_model).state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "target_mean": target_mean,
        "target_std": target_std,
        "position_tokens": TOKEN_TABLES.get("position_tokens", None),
        "trajectory_tokens": TOKEN_TABLES.get("trajectory_tokens", None),
        "current_time_index": CURRENT_TIME_INDEX,
        "history_steps": HISTORY_STEPS,
        "future_steps": FUTURE_STEPS,
        "total_scene_steps": TOTAL_SCENE_STEPS,
        "neighbors_k": NEIGHBORS_K,
        "diffusion_t": DIFFUSION_T,
        "sample_steps": DIFFUSION_SAMPLE_STEPS,
        "guidance_scale": GUIDANCE_SCALE,
        "ema_decay": EMA_DECAY,
        "history": history if "history" in globals() else [],
    },
    SAVE_PATH,
)
print(f"Checkpoint saved to {SAVE_PATH}")
print(f"Latest resumable training checkpoint: {TRAINING_CHECKPOINT_PATH}")
print(f"Best EMA checkpoint: {BEST_EMA_CHECKPOINT_PATH}")
